In [1]:
import requests
from bs4 import BeautifulSoup
from dataclasses import dataclass
from typing import Optional, Dict, List
import csv
import time
from urllib.parse import urljoin
import pandas as pd


# Target
# So my targer is scraping quotes, authors, and tags from https://quotes.toscrape.com
# to collect structured textual data for analysis and demonstration of web scraping


BASE_URL = "https://quotes.toscrape.com/"

#Fetcher
def fetch_html(url: str, headers: Optional[Dict[str, str]] = None, timeout_s: float = 15.0) -> str:
    """Fetch raw HTML from a given URL."""
    if headers is None:
        # I use a common browser User-Agent to avoid detection
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}
    resp = requests.get(url, headers=headers, timeout=timeout_s)
    resp.raise_for_status()
    return resp.text


# Dataclass
@dataclass
class Quote:
    text: str
    author: str
    tags: List[str]


# Parser
def parse_quotes(html: str) -> List[Quote]:
    """Extract quote info (text, author, tags) from HTML."""
    soup = BeautifulSoup(html, "html.parser")
    quotes = []
    for quote_div in soup.select("div.quote"):
        text = quote_div.select_one("span.text").text
        author = quote_div.select_one("small.author").text
        tags = [tag.text for tag in quote_div.select("div.tags a.tag")]
        quotes.append(Quote(text, author, tags))
    return quotes


# Pagination
def scrape_quotes(max_pages: int = 10, sleep_s: float = 2.0) -> List[Quote]:
    """Follow pagination until max_pages reached."""
    all_quotes = []
    page = 1
    current_url = BASE_URL

    while current_url and page <= max_pages:
        print(f"Scraping page {page}...")
        try:
            html = fetch_html(current_url)
        except requests.exceptions.RequestException as e:
            print(f"Error fetching {current_url}: {e}")
            # Stop scraping if an error occurs
            break

        quotes = parse_quotes(html)
        all_quotes.extend(quotes)

        soup = BeautifulSoup(html, "html.parser")
        next_link = soup.select_one("li.next a")
        if next_link:
          # Use urljoin to handle relative URLs correctly
          current_url = urljoin(BASE_URL, next_link["href"])
        else:
          current_url = None

        # Increased sleep time
        time.sleep(sleep_s)
        page += 1

    return all_quotes


# Run scraper
quotes = scrape_quotes()
print(f"Collected {len(quotes)} quotes.")

# Display the collected data as a DataFrame
quotes_df = pd.DataFrame([quote.__dict__ for quote in quotes])
display(quotes_df)


# CSV export
csv_path = "quotes_scraped.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f, delimiter=';')
    writer.writerow(["Text", "Author", "Tags"])
    for q in quotes:
        writer.writerow([q.text, q.author, ", ".join(q.tags)])

print(f"Saved to {csv_path}")
#Documentation: well the site structure was clean and predictable.
# So quotes, authors and tags were easy to locate using CSS selectors.
# No issues encountered and pagination and HTML parsing worked as expected
# What if we speak about handling, I added error handling for requests
# and also like a polite delay (`sleep_s=2`) between requests

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
Collected 100 quotes.


,text,author,tags
0,“The world as we have created it is a process ...,Albert Einstein,"[change, deep-thoughts, thinking, world]"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"[abilities, choices]"
2,“There are only two ways to live your life. On...,Albert Einstein,"[inspirational, life, live, miracle, miracles]"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"[aliteracy, books, classic, humor]"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"[be-yourself, inspirational]"
...,...,...,...
95,“You never really understand a person until yo...,Harper Lee,[better-life-empathy]
96,“You have to write the book that wants to be w...,Madeleine L'Engle,"[books, children, difficult, grown-ups, write,..."
97,“Never tell the truth to people who are not wo...,Mark Twain,[truth]
98,"“A person's a person, no matter how small.”",Dr. Seuss,[inspirational]


Saved to quotes_scraped.csv
